In [31]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root))

In [32]:
from qiskit.circuit import QuantumCircuit,ParameterVector
from qiskit_machine_learning.kernels import FidelityQuantumKernel
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,recall_score,f1_score,classification_report
from src.data_load import load_data
from src.preprocess import preprocess

In [33]:
seed=42
qubits=4

X_train, X_test, y_train, y_test= load_data()
X_train_scale,X_test_scale, X_train_pca, X_test_pca, pca=preprocess(X_train,X_test,qubits)

In [34]:
mm = MinMaxScaler(feature_range=(0, np.pi))
X_train_quantum = mm.fit_transform(X_train_pca)
X_test_quantum = np.clip(mm.transform(X_test_pca), 0, np.pi)

In [35]:
x=ParameterVector("x",qubits)
feature_map=QuantumCircuit(qubits)
for i in range(qubits):
    feature_map.ry(x[i],i)

In [36]:
kernel=FidelityQuantumKernel(feature_map=feature_map)
K_train = kernel.evaluate(X_train_quantum, X_train_quantum)
K_test = kernel.evaluate(X_test_quantum, X_train_quantum)
print(K_train.shape,K_test.shape)

(455, 455) (114, 455)


In [38]:
qsvm=SVC(kernel="precomputed",C=1.0)
qsvm.fit(K_train, y_train)
y_pred_q=qsvm.predict(K_test)

In [39]:
print("QSVM accuracy:", accuracy_score(y_test, y_pred_q))
print("Malignant recall:", recall_score(y_test, y_pred_q, pos_label=0))
print("F1 (macro):", f1_score(y_test, y_pred_q, average="macro"))
print(classification_report(y_test, y_pred_q, target_names=["malignant", "benign"]))

QSVM accuracy: 0.956140350877193
Malignant recall: 0.9047619047619048
F1 (macro): 0.9521289997480473
              precision    recall  f1-score   support

   malignant       0.97      0.90      0.94        42
      benign       0.95      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



In [40]:
svm = SVC(kernel="rbf", C=1.0, gamma="scale").fit(X_train_quantum, y_train)
y_pred_classical = svm.predict(X_test_quantum)
print("Classical SVM (same data) accuracy:", accuracy_score(y_test, y_pred_classical))
print("Classical malignant recall:", recall_score(y_test, y_pred_classical, pos_label=0))
print("Classical F1 (macro):", f1_score(y_test, y_pred_classical, average="macro"))

Classical SVM (same data) accuracy: 0.956140350877193
Classical malignant recall: 0.9285714285714286
Classical F1 (macro): 0.9526381387619444
